In [1]:
import cv2
import numpy as np
import torch
from depth_anything_v2.dpt import DepthAnythingV2
import vtk
from PIL import Image


xFormers not available
xFormers not available


In [69]:
import os
model = DepthAnythingV2(encoder='vitb', features=128, out_channels=[96, 192, 384, 768])
model.load_state_dict(torch.load('./checkpoints/depth_anything_v2_vitb.pth', map_location='cuda:0')) #cpu


# model = DepthAnythingV2(encoder='vits', features=64, out_channels=[48, 96, 192, 384])
# model.load_state_dict(torch.load('./checkpoints/depth_anything_v2_vits.pth', map_location='cuda:0'))

def pic23d(pic_path,save_path):

    if ".png"in pic_path or ".webp" in pic_path:
        image = Image.open(pic_path)  # 或者 "example.png"

        # 转换并保存为 JPG 格式
        pic_path = pic_path.replace(".webp",".jpg").replace(".png",".jpg")
        image.convert("RGB").save(pic_path, "JPEG")



    model.cuda().eval()   #   cpu model.eval() 

    raw_img = cv2.imread(pic_path)  
    height, width = raw_img.shape[:2]  
    target_height = 600  
    scale_ratio = target_height / height  
    target_width = int(width * scale_ratio)  
    raw_img = cv2.resize(raw_img, (target_width, target_height))  


    depth = model.infer_image(raw_img) # HxW raw depth map



    depth_normalized = cv2.normalize(depth, None, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)  
    depth_display = (depth_normalized * 255).astype(np.uint8)  # 转换为8位图像以便显示  
    cv2.imwrite('./depth_map.png', depth_display)  

    fx = 50  # 焦距x  
    fy = 50# 焦距y  
    cx = raw_img.shape[1] / 2.0  # 光心x  
    cy = raw_img.shape[0] / 2.0  # 光心y  
    height, width = raw_img.shape[:2]  
    
    # 推断深度图  
    depth_ = depth_display/255
    # 深度图可能需要归一化或转换到实际的深度值，这里假设它已经是所需的格式  
    
    # 创建一个空的数组来存储空间位置坐标  
    pos = np.zeros((height * width, 3), dtype=np.float32)  
    d_list = []
    # 遍历深度图的每个像素  
    for v in range(height):  
        for u in range(width):  
            d = depth_[v, u]  # 获取深度值  
            # 计算空间坐标  
            X = (1-0.5*d)*(u - cx) * 1/ fx  
            Y = (1-0.5*d)*(v - cy)  * 1/ fy  
            Z = d  *10
            d_list.append(d)
            
            # 将坐标存储到pos数组中  
            idx = v * width + u  
            pos[idx] = [X, Y, Z]  
    
    # 现在pos数组包含了深度图上每个点的空间位置坐标
 


    with open("a.obj","w") as t:
        t.writelines("mtllib my_mtl.mtl"+"\n")
            
        for l in pos:
            L = "v "+str(l[0])+" "+str(l[1])+" "+str(l[2])+"\n"
            t.writelines(L )
        
        w_x = np.asarray([np.linspace(0.0, 1.0, num=width)]*height).reshape(width*height)
        h_y = np.asarray([np.linspace(1.0, 0.0, num=height)]*width).T.reshape(width*height)
        t_array_ = np.stack([w_x,h_y],axis=1)


        for i,j in t_array_:
            L = "vt " +str(i)+" "+str(j) +"\n"
            t.writelines(L )                
    #  
    #          
        t.writelines("\nusemtl my_mtl\n" )              
        b = np.array(range(height * width)).reshape([ height ,width])
        print(b.shape)  
                
        for i in range(1,b.shape[0]-1):
            for j in range(1,b.shape[1]-1):
                p = [ b[i,j] , b[i+1,j],b[i+1,j+1],b[i,j+1]]
        
                L = "f "+str(p[0])+"/"+ str(p[0])+" "+str(p[1])+"/"+ str(p[1])+" "+str(p[2])+"/"+ str(p[2])+" "+str(p[3])+"/"+ str(p[3])+"\n"
                t.writelines(L )
            


    with open("my_mtl.mtl","w") as m:  
        m.writelines(
    f"""
    newmtl my_mtl
    Ka 1 1 1
    Kd 1 1 1
    d 1
    Ns 0
    illum 1
    map_Kd {pic_path}
    """
        )



    image_size_x,image_size_y = raw_img.shape[:2] 
    def render_and_save_images(obj_file,   image_size=(int(image_size_x*1.6),int(image_size_y*1.6) ) ):
        # 读取模型
        reader = vtk.vtkOBJReader()
        reader.SetFileName(obj_file)
        reader.Update()

        # 创建一个映射器
        mapper = vtk.vtkPolyDataMapper()
        mapper.SetInputConnection(reader.GetOutputPort())

        # 创建一个演员
        actor = vtk.vtkActor()
        actor.SetMapper(mapper)

        # 读取贴图文件
        texture = vtk.vtkTexture()
        texture_reader = vtk.vtkJPEGReader()  # 假设贴图为JPEG格式
        texture_reader.SetFileName(pic_path)  # 根据.mtl文件中的路径设置
        texture_reader.Update()
        texture.SetInputConnection(texture_reader.GetOutputPort())
        
        # 将贴图应用到演员
        actor.SetTexture(texture)
        actor.SetOrientation(1,0,0)

        # 创建渲染窗口
        render_window = vtk.vtkRenderWindow()
        render_window.SetSize(image_size[1], image_size[0])  # 设置渲染窗口大小 

        # 创建左右视角的渲染器
        total_pic_count = 40
        renderers = []
        image_sequence = []
        for i in range(total_pic_count):
            renderer = vtk.vtkRenderer()
            render_window.AddRenderer(renderer)
            renderers.append(renderer)
            # 禁用光照
            renderer.SetAmbient(100.0, 100.0, 100.0)  # 设置环境光
            actor.GetProperty().SetLighting(False)
            # 设置摄像机
            camera = vtk.vtkCamera()
            
            camera.SetFocalPoint(0, 0, 0)
            # camera.SetParallelProjection(True)
            # # 正交投影参数
            # parallel_scale = 5.0  # 根据需求调整正交比例
            # camera.SetParallelScale(parallel_scale)
      
            camera.SetPosition(-4+i*0.2, 0, 20)  # 可以根据需要调整位置
            camera.SetViewUp(0, -1, 0)

            renderer.SetActiveCamera(camera)

            # 添加演员到渲染器
            renderer.AddActor(actor)
            renderer.SetBackground(0.0, 0.0, 0.0)  # 背景颜色

            # 渲染
            render_window.Render()

            # 保存图像
            window_to_image_filter = vtk.vtkWindowToImageFilter()
            window_to_image_filter.SetInput(render_window)
            window_to_image_filter.ReadFrontBufferOff()  # 读取后台缓冲区
            window_to_image_filter.Update()

            writer = vtk.vtkPNGWriter()
            image_path = f"./output/{i:04d}.png"

            if not os.path.exists('./output'):
                os.makedirs('./output')
            writer.SetFileName(image_path)
            image_sequence.append(image_path)
       
        
            writer.SetInputConnection(window_to_image_filter.GetOutputPort())
            writer.Write()

        

            
        # 获取视频的帧宽和帧高
        frame_width = image_size[1]
        frame_height = image_size[0]
        
        # 视频编码设置
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 使用mp4v编码
        output_video = "output.mp4"
        video_writer = cv2.VideoWriter(output_video, fourcc, 30, (frame_width, frame_height))

        # 读取并写入每一帧
        for image_path in image_sequence:
            img = cv2.imread(image_path)
            video_writer.write(img)
        
        # 释放视频写入对象
        video_writer.release()
    

    render_and_save_images('a.obj' )


pic_path = "./output_images/frame_0002.png"
save_path = "./output/frame.png"
pic23d(pic_path,save_path)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_46740\2117098010.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./checkpoints/dept

(600, 415)


In [ ]:
import cv2
import os

def save_frames_from_video(video_path, output_folder, num_frames_range=[0,50]):
    # 创建输出文件夹
    os.makedirs(output_folder, exist_ok=True)

    # 打开视频文件
    cap = cv2.VideoCapture(video_path)

    # 检查视频是否成功打开
    if not cap.isOpened():
        print("无法打开视频文件")
        return

    frame_count = 0


    for i in range(0,36):
        save_path = f"C:/Users/Administrator/Pictures/blender_gif/{i+1}/"
        os.makedirs(save_path, exist_ok=True)


    pic_count = 0
    while True:
        ret, frame = cap.read()  # 读取帧
        if not ret:
            break  # 如果没有更多帧，退出循环
        frame_count += 1
        if  (frame_count >= num_frames_range[0] and frame_count < num_frames_range[1]) :
            pic_count+=1

            # 保存帧为图像文件
            frame_filename = os.path.join(output_folder, f'{pic_count}.jpg')
            cv2.imwrite(frame_filename, frame)
            

            save_path = frame_filename.replace("blender_gif/","blender_gif/_/")

            pic23d(frame_filename,save_path)
   



    
    cap.release()  # 释放视频捕获对象
    print(f'已保存 {frame_count} 帧到 {output_folder}')



# 使用示例
video_file = './video.mp4'  # 视频文件路径
output_dir = 'C:/Users/Administrator/Pictures/blender_gif/'    # 输出文件夹路径
save_frames_from_video(video_file, output_dir)

(600, 337)
(600, 337)


KeyboardInterrupt: 

In [3]:
video_file = './CogVideoX_00013.mp4'  # 视频文件路径
output_dir = 'C:/Users/Administrator/Pictures/blender_gif12/'    # 输出文件夹路径
save_frames_from_video(video_file, output_dir)

NameError: name 'pic23d' is not defined

In [21]:
import requests
from PIL import Image
from io import BytesIO

def upload_pic(pic_file_name, save_base="index.bin"):
    im = Image.open(pic_file_name)
    
    # Adjust the image size while maintaining the aspect ratio
    original_width, original_height = im.size
    target_height = 240
    target_width = int((target_height / original_height) * original_width)
    im = im.resize((target_width, target_height))  # Use LANCZOS filter for better quality

    # Optionally, convert to RGB if the image has an alpha channel
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        im = im.convert("RGB")

    # Save as a binary file (using JPEG with adjustable quality)
    img_byte = BytesIO()
    im.save(img_byte, format='JPEG', quality=100)  # Adjust quality as needed
    img_data = img_byte.getvalue()

    # Write to binary file
    with open(save_base, "wb") as f:
        f.write(img_data)



In [43]:
import os
pic_path = "C:/Users/Administrator/Pictures/blender_gif/"


for i in range(0,72):
    for pic in os.listdir(pic_path+"/"+str(i)):
        if ".jpg" in pic or ".png" in pic:
            # print(pic_path+pic)
            if not os.path.exists("F:/move2/"+str(i)+"/"):
                os.mkdir("F:/move2/"+str(i)+"/")
            upload_pic(pic_path+"/"+str(i)+"/"+pic,"F:/move2/"+str(i)+"/"+str(int(pic.split(".")[0]))+".bin")

In [ ]:
import bpy
import os

# 设置保存路径



# 获取场景中的Cube对象
cube = bpy.data.objects.get("圆环")
if not cube:
    raise ValueError("找不到名为 'Cube' 的对象，请确保它存在")

# 配置渲染设置
bpy.context.scene.render.image_settings.file_format = 'PNG'  # 渲染为PNG格式
bpy.context.scene.render.resolution_x = 800  # 宽度
bpy.context.scene.render.resolution_y = 800  # 高度

# 旋转和渲染逻辑
for index, i in enumerate( list( range(-180, 180, 5))):
    
    save_path = f"C:/Users/Administrator/Pictures/blender_gif/{index}/"
    os.makedirs(save_path, exist_ok=True)
    
    # 旋转Cube对象
    cube.rotation_euler[2] = i * (3.14159 / 180)  # 将角度转换为弧度
    
    for frame in range(1,43):
    
        # 设置帧号（用于保存文件名）
        bpy.context.scene.frame_set(frame)  # 每5度为一帧
    
        # 设置输出文件路径
        file_name = f"{frame}.png"
        bpy.context.scene.render.filepath = os.path.join(save_path, file_name)
    
        # 渲染当前帧
        #bpy.ops.render.render(write_still=True)
        bpy.ops.render.opengl(write_still=True, view_context=True)
    index+=1
  

print("渲染完成！所有图片保存在：", save_path)

In [22]:
import cv2
import os
import numpy as np

def resize_frame(frame, target_size=240):
    """
    等比例缩放图像，将其最长边缩放到指定大小。

    :param frame: 输入的图像帧
    :param target_size: 目标大小，最长边将缩放到该大小
    :return: 缩放后的图像
    """
    # 获取图像的宽度和高度
    height, width = frame.shape[:2]
    
    # 选择最长边的尺寸
    if width > height:
        new_width = target_size
        new_height = int(target_size * height / width)
    else:
        new_height = target_size
        new_width = int(target_size * width / height)

    # 调整图像大小
    resized_frame = cv2.resize(frame, (new_width, new_height))
    return resized_frame

def place_on_black_background(frame, target_size=240):
    """
    将缩放后的图像居中放置在黑色背景上。

    :param frame: 输入的缩放后的图像
    :param target_size: 背景图像的目标大小（240x240）
    :return: 居中放置后的图像
    """
    # 创建黑色背景
    background = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    # 获取缩放后图像的尺寸
    height, width = frame.shape[:2]

    # 计算图像放置的位置（居中）
    y_offset = (target_size - height) // 2
    x_offset = (target_size - width) // 2

    # 将图像放置到黑色背景的中心
    background[y_offset:y_offset + height, x_offset:x_offset + width] = frame

    return background

def extract_frames_from_video(video_path, output_folder='frames_output', target_size=240):
    """
    提取视频中的每一帧，缩放并将其居中放置在黑色背景上，然后保存到指定的子文件夹中。

    :param video_path: 输入的视频文件路径
    :param output_folder: 存储提取帧的根文件夹
    :param target_size: 目标尺寸，最长边将缩放到该大小
    """
    # 创建输出文件夹（如果不存在的话）
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 打开视频文件
    cap = cv2.VideoCapture(video_path)

    # 检查视频是否成功打开
    if not cap.isOpened():
        print(f"无法打开视频文件: {video_path}")
        return

    # 获取视频的总帧数
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_index = 0

    while True:
        # 读取一帧
        ret, frame = cap.read()
        
        # 如果没有读取到帧，说明视频结束
        if not ret:
            break

        # 缩放图像
        resized_frame = resize_frame(frame, target_size)

        # 将缩放后的图像居中放置在黑色背景上
        final_frame = place_on_black_background(resized_frame, target_size)

        # 为每一帧创建子文件夹
        frame_folder = os.path.join(output_folder, f"{frame_index}")
        if not os.path.exists(frame_folder):
            os.makedirs(frame_folder)

        # 保存每一帧到子文件夹
        frame_filename = os.path.join(frame_folder, f"{frame_index}.jpg")
        cv2.imwrite(frame_filename, final_frame)
        
        print(f"Frame {frame_index + 1}/{frame_count} saved to {frame_filename}")
        
        # 增加帧的索引
        frame_index += 1

    # 释放视频对象
    cap.release()

    print("视频帧提取、缩放并居中完成！")


# 示例使用
video_path = 'CogVideoX_Fun_orbits_00005.mp4'
extract_frames_from_video(video_path)


Frame 1/66 saved to frames_output\0\0.jpg
Frame 2/66 saved to frames_output\1\1.jpg
Frame 3/66 saved to frames_output\2\2.jpg
Frame 4/66 saved to frames_output\3\3.jpg
Frame 5/66 saved to frames_output\4\4.jpg
Frame 6/66 saved to frames_output\5\5.jpg
Frame 7/66 saved to frames_output\6\6.jpg
Frame 8/66 saved to frames_output\7\7.jpg
Frame 9/66 saved to frames_output\8\8.jpg
Frame 10/66 saved to frames_output\9\9.jpg
Frame 11/66 saved to frames_output\10\10.jpg
Frame 12/66 saved to frames_output\11\11.jpg
Frame 13/66 saved to frames_output\12\12.jpg
Frame 14/66 saved to frames_output\13\13.jpg
Frame 15/66 saved to frames_output\14\14.jpg
Frame 16/66 saved to frames_output\15\15.jpg
Frame 17/66 saved to frames_output\16\16.jpg
Frame 18/66 saved to frames_output\17\17.jpg
Frame 19/66 saved to frames_output\18\18.jpg
Frame 20/66 saved to frames_output\19\19.jpg
Frame 21/66 saved to frames_output\20\20.jpg
Frame 22/66 saved to frames_output\21\21.jpg
Frame 23/66 saved to frames_output\22\2

In [20]:
from moviepy import VideoFileClip, ImageSequenceClip
from PIL import Image
import numpy as np

# 定义屏幕宽高
screen_width = 368
screen_height = 552

# 输入和输出视频文件路径
input_video_path = 'CogVideoX_Fun_orbits_00005.mp4'
output_video_path = 'output_video.mp4'

# 设置模式：1表示原来处理方式，2表示避免黑边的处理方式
mode = 2  # 设置为1或2

def process_frame(frame, mode):
    """处理每一帧：根据模式进行处理"""
    # 转换为Pillow图像对象
    pil_image = Image.fromarray(frame)
    
    # 获取原始视频帧的宽高
    original_width, original_height = pil_image.size

    if mode == 1:
        # 模式1: 缩放帧，保持宽高比例，再裁剪到目标尺寸（可能会有黑边）
        pil_image = pil_image.resize((int(original_width * min(screen_width / original_width, screen_height / original_height)), 
                                      int(original_height * min(screen_width / original_width, screen_height / original_height))))
        
        # 计算裁剪区域，按中心裁剪
        left = (pil_image.width - screen_width) / 2
        top = (pil_image.height - screen_height) / 2
        right = (pil_image.width + screen_width) / 2
        bottom = (pil_image.height + screen_height) / 2
        
        # 裁剪图像
        pil_image = pil_image.crop((left, top, right, bottom))
        
    elif mode == 2:
        # 模式2: 按屏幕宽高比裁剪视频，避免黑边
        screen_ratio = screen_width / screen_height
        video_ratio = original_width / original_height

        if video_ratio > screen_ratio:
            # 视频宽高比大于屏幕宽高比，裁剪宽度
            new_width = int(original_height * screen_ratio)
            left = (original_width - new_width) / 2
            pil_image = pil_image.crop((left, 0, left + new_width, original_height))
        elif video_ratio < screen_ratio:
            # 视频宽高比小于屏幕宽高比，裁剪高度
            new_height = int(original_width / screen_ratio)
            top = (original_height - new_height) / 2
            pil_image = pil_image.crop((0, top, original_width, top + new_height))
        # 如果视频宽高比等于屏幕宽高比，则不做裁剪，直接缩放
        
        # 缩放到目标尺寸
        pil_image = pil_image.resize((screen_width, screen_height))

    # 转换回NumPy数组
    return np.array(pil_image)

# 加载输入视频
clip = VideoFileClip(input_video_path)

# 使用 iter_frames 获取每一帧并处理
frames = []
for frame in clip.iter_frames(fps=clip.fps, dtype='uint8'):
    processed_frame = process_frame(frame, mode)
    frames.append(processed_frame)

# 使用处理后的帧生成新的视频
processed_clip = ImageSequenceClip(frames, fps=clip.fps)

# 输出处理后的视频
processed_clip.write_videofile(output_video_path, codec="libx264", fps=clip.fps)


MoviePy - Building video output_video.mp4.
MoviePy - Writing video output_video.mp4



MoviePy - Done !
MoviePy - video ready output_video.mp4


In [29]:
!pip show moviepy

'pip' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


In [26]:
import cv2
import os
import numpy as np

# 配置
video_path = "input_video.mp4"  # 输入视频路径
output_folder = "framebuffer_frames"  # 输出文件夹
screen_width = int(368)
screen_height = int(552)

# 创建输出文件夹
os.makedirs(output_folder, exist_ok=True)

# 打开视频文件
cap = cv2.VideoCapture(video_path)

frame_index = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 调整帧大小到屏幕分辨率
    frame = cv2.resize(frame, (screen_width, screen_height))

    # 转换颜色空间 BGR -> BGRA
    frame_bgra = cv2.cvtColor(frame, cv2.COLOR_BGR2BGRA)

    # 保存为 framebuffer.raw 文件
    raw_file_path = os.path.join(output_folder, f"{frame_index:05d}.raw")
    with open(raw_file_path, "wb") as raw_file:
        raw_file.write(frame_bgra.tobytes())

    print(f"Saved frame {frame_index} as {raw_file_path}")
    frame_index += 1

cap.release()
print(f"Finished extracting frames. Total frames: {frame_index}")

Saved frame 0 as framebuffer_frames\00000.raw
Saved frame 1 as framebuffer_frames\00001.raw
Saved frame 2 as framebuffer_frames\00002.raw
Saved frame 3 as framebuffer_frames\00003.raw
Saved frame 4 as framebuffer_frames\00004.raw
Saved frame 5 as framebuffer_frames\00005.raw
Saved frame 6 as framebuffer_frames\00006.raw
Saved frame 7 as framebuffer_frames\00007.raw
Saved frame 8 as framebuffer_frames\00008.raw
Saved frame 9 as framebuffer_frames\00009.raw
Saved frame 10 as framebuffer_frames\00010.raw
Saved frame 11 as framebuffer_frames\00011.raw
Saved frame 12 as framebuffer_frames\00012.raw
Saved frame 13 as framebuffer_frames\00013.raw
Saved frame 14 as framebuffer_frames\00014.raw
Saved frame 15 as framebuffer_frames\00015.raw
Saved frame 16 as framebuffer_frames\00016.raw
Saved frame 17 as framebuffer_frames\00017.raw
Saved frame 18 as framebuffer_frames\00018.raw
Saved frame 19 as framebuffer_frames\00019.raw
Saved frame 20 as framebuffer_frames\00020.raw
Saved frame 21 as frame

In [29]:
from moviepy import VideoFileClip, ImageSequenceClip
from PIL import Image
import numpy as np

# 定义屏幕宽高
screen_width = 368
screen_height = 552

# 输入和输出视频文件路径
input_video_path = 'your_video (2).mp4'
output_video_path = 'output_video.mp4'

# 设置模式：1表示原来处理方式，2表示避免黑边的处理方式
mode = 2  # 设置为1或2

def process_frame(frame, mode):
    """处理每一帧：根据模式进行处理"""
    # 转换为Pillow图像对象
    pil_image = Image.fromarray(frame)
    
    # 获取原始视频帧的宽高
    original_width, original_height = pil_image.size

    if mode == 1:
        # 模式1: 缩放帧，保持宽高比例，再裁剪到目标尺寸（可能会有黑边）
        pil_image = pil_image.resize((int(original_width * min(screen_width / original_width, screen_height / original_height)), 
                                      int(original_height * min(screen_width / original_width, screen_height / original_height))))
        
        # 计算裁剪区域，按中心裁剪
        left = (pil_image.width - screen_width) / 2
        top = (pil_image.height - screen_height) / 2
        right = (pil_image.width + screen_width) / 2
        bottom = (pil_image.height + screen_height) / 2
        
        # 裁剪图像
        pil_image = pil_image.crop((left, top, right, bottom))
        
    elif mode == 2:
        # 模式2: 按屏幕宽高比裁剪视频，避免黑边
        screen_ratio = screen_width / screen_height
        video_ratio = original_width / original_height

        if video_ratio > screen_ratio:
            # 视频宽高比大于屏幕宽高比，裁剪宽度
            new_width = int(original_height * screen_ratio)
            left = (original_width - new_width) / 2
            pil_image = pil_image.crop((left, 0, left + new_width, original_height))
        elif video_ratio < screen_ratio:
            # 视频宽高比小于屏幕宽高比，裁剪高度
            new_height = int(original_width / screen_ratio)
            top = (original_height - new_height) / 2
            pil_image = pil_image.crop((0, top, original_width, top + new_height))
        # 如果视频宽高比等于屏幕宽高比，则不做裁剪，直接缩放
        
        # 缩放到目标尺寸
        pil_image = pil_image.resize((screen_width, screen_height))

    # 转换回NumPy数组
    return np.array(pil_image)

# 加载输入视频
clip = VideoFileClip(input_video_path)

# 使用 iter_frames 获取每一帧并处理
frames = []
for frame in clip.iter_frames(fps=clip.fps, dtype='uint8'):
    processed_frame = process_frame(frame, mode)
    frames.append(processed_frame)

# 使用处理后的帧生成新的视频
processed_clip = ImageSequenceClip(frames, fps=clip.fps)

# 输出处理后的视频
processed_clip.write_videofile(output_video_path, codec="libx264", fps=clip.fps)


MoviePy - Building video output_video.mp4.
MoviePy - Writing video output_video.mp4



MoviePy - Done !
MoviePy - video ready output_video.mp4


In [30]:
import cv2
import os

# 视频文件路径
video_file = "output_video.mp4"
output_folder = "video_frames"

# 创建存放视频帧的文件夹
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 打开视频文件
cap = cv2.VideoCapture(video_file)

# 获取视频的总帧数
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if not cap.isOpened():
    print("Error: Couldn't open video file.")
    exit()

frame_num = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break  # 如果读取不到帧，则退出

    # 构造文件路径
    frame_filename = os.path.join(output_folder, f"frame_{frame_num:04d}.jpg")

    # 保存每帧为jpg文件
    cv2.imwrite(frame_filename, frame)
    
    frame_num += 1

cap.release()
print(f"Finished processing {frame_num} frames.")

Finished processing 141 frames.


In [68]:
import cv2
import os

# 输入视频文件路径
video_path = 'input_video.mp4'
# 输出图片保存目录
output_dir = 'output_images/'

# 创建保存图片的目录，如果不存在
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 打开视频文件
cap = cv2.VideoCapture(video_path)

# 获取视频的帧率
fps = cap.get(cv2.CAP_PROP_FPS)
# 获取视频的总帧数
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frame_index = 0

# 读取每一帧
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break  # 如果读取不到帧，则退出

    # 构建输出图片的文件名
    output_file = os.path.join(output_dir, f"frame_{frame_index:04d}.png")
    
    # 保存当前帧为 PNG 图片
    cv2.imwrite(output_file, frame)
    
    print(f"保存帧 {frame_index + 1}/{frame_count} 到 {output_file}")

    frame_index += 1

# 释放视频对象
cap.release()
print("转换完成")


保存帧 1/141 到 output_images/frame_0000.png
保存帧 2/141 到 output_images/frame_0001.png
保存帧 3/141 到 output_images/frame_0002.png
保存帧 4/141 到 output_images/frame_0003.png
保存帧 5/141 到 output_images/frame_0004.png
保存帧 6/141 到 output_images/frame_0005.png
保存帧 7/141 到 output_images/frame_0006.png
保存帧 8/141 到 output_images/frame_0007.png
保存帧 9/141 到 output_images/frame_0008.png
保存帧 10/141 到 output_images/frame_0009.png
保存帧 11/141 到 output_images/frame_0010.png
保存帧 12/141 到 output_images/frame_0011.png
保存帧 13/141 到 output_images/frame_0012.png
保存帧 14/141 到 output_images/frame_0013.png
保存帧 15/141 到 output_images/frame_0014.png
保存帧 16/141 到 output_images/frame_0015.png
保存帧 17/141 到 output_images/frame_0016.png
保存帧 18/141 到 output_images/frame_0017.png
保存帧 19/141 到 output_images/frame_0018.png
保存帧 20/141 到 output_images/frame_0019.png
保存帧 21/141 到 output_images/frame_0020.png
保存帧 22/141 到 output_images/frame_0021.png
保存帧 23/141 到 output_images/frame_0022.png
保存帧 24/141 到 output_images/frame_0023.png
保